In [12]:
#!/usr/bin/env Rscript
#
# scATAC-seq Single Sample Processing (完整流程 - 按照师姐的方法)
#
# Usage:
#   Rscript process_single_sample.R --gse GSE283746 --gsm GSM8671454 --fragment-file data/raw/GSE283746/GSM8671454_pHC8_snATAC_fragments.tsv.gz

suppressPackageStartupMessages({
  library(optparse)
  library(Signac)
  library(Seurat)
  library(GenomeInfoDb)
  library(EnsDb.Hsapiens.v86)
  library(scDblFinder)
  library(SingleCellExperiment)
  library(Matrix)
  library(rtracklayer)
})


In [13]:

# 解析命令行参数
setwd("/Users/kanae/Works/ML2026")
option_list <- list(
  make_option(c("--gse"), type = "character", help = "GSE ID"),
  make_option(c("--gsm"), type = "character", help = "GSM ID"),
  make_option(c("--fragment-file"), type = "character", help = "Fragment 文件路径 (可选)"),
  make_option(c("--peak-file"), type = "character", default = "data/reference/peak.bed", help = "Peak 文件"),
  make_option(c("--output-dir"), type = "character", help = "输出目录"),
  make_option(c("--genome"), type = "character", default = "hg38", help = "基因组版本"),
  make_option(c("--nmads"), type = "numeric", default = 4, help = "MAD 倍数"),
  make_option(c("--skip-matrix"), action = "store_true", default = FALSE, help = "跳过矩阵生成，从已有RDS加载")
)

opt_parser <- OptionParser(option_list = option_list)

opt <- parse_args(
  opt_parser,
  args = c(
    "--gse", "GSE283744",
    "--gsm", "GSM8671454"
  )
)

if (is.null(opt$gse) || is.null(opt$gsm)) {
  print_help(opt_parser)
  stop("必须提供 --gse, --gsm 参数")
}

if (is.null(opt$`output-dir`)) {
  opt$`output-dir` <- file.path("output", opt$gse)
}

if (is.null(opt$`fragment-file`)) {
  raw_dir <- file.path("data/raw", opt$gse)
  fragment_files <- list.files(raw_dir, pattern = paste0("^", opt$gsm, "_.*_fragments\\.tsv\\.gz$"), full.names = TRUE)
  if (length(fragment_files) == 0) {
    stop("未找到 fragment 文件: ", file.path(raw_dir, paste0(opt$gsm, "_*_fragments.tsv.gz")))
  }
  opt$`fragment-file` <- fragment_files[1]
}

cat(rep("=", 80), "\n", sep = "")
cat("scATAC-seq Single Sample Processing\n")
cat(rep("=", 80), "\n", sep = "")
cat("GSE ID:", opt$gse, "\n")
cat("GSM ID:", opt$gsm, "\n")
cat("Fragment file:", opt$`fragment-file`, "\n")
cat("Peak file:", opt$`peak-file`, "\n")
cat("Output directory:", opt$`output-dir`, "\n\n")

dir.create(opt$`output-dir`, recursive = TRUE, showWarnings = FALSE)


scATAC-seq Single Sample Processing
GSE ID: GSE283744 
GSM ID: GSM8671454 
Fragment file: data/raw/GSE283744/GSM8671454_pHC8_snATAC_fragments.tsv.gz 
Peak file: data/reference/peak.bed 
Output directory: output/GSE283744 



In [14]:

# 检查是否跳过矩阵生成
if (opt$`skip-matrix`) {
  cat("\n跳过矩阵生成，从已有 RDS 加载...\n")
  rds_raw <- file.path(opt$`output-dir`, paste0(opt$gsm, "_seurat_raw.rds"))
  if (!file.exists(rds_raw)) {
    stop("未找到原始 RDS 文件: ", rds_raw)
  }
  atac_obj <- readRDS(rds_raw)
  fragment_file <- opt$`fragment-file`
  if (is.null(fragment_file)) {
    raw_dir <- file.path("data/raw", opt$gse)
    fragment_files <- list.files(raw_dir, pattern = paste0("^", opt$gsm, "_.*_fragments\\.tsv\\.gz$"), full.names = TRUE)
    fragment_file <- fragment_files[1]
  }
  cat("加载完成，细胞数:", ncol(atac_obj), "\n\n")
} else {
  cat("不跳过矩阵生成，继续从 fragment 构建对象...\n")
}


# MAD 方法
is_outlier <- function(dataframe, metric = NULL, nmads = NULL) {
  if (!metric %in% colnames(dataframe)) {
    stop("metric must be a valid column name")
  }
  M <- dataframe[, metric]
  median_val <- median(M, na.rm = TRUE)
  mad_val <- mad(M, na.rm = TRUE)
  outlier <- (M < median_val - nmads * mad_val) | (median_val + nmads * mad_val < M)
  return(outlier)
}


不跳过矩阵生成，继续从 fragment 构建对象...


In [15]:
cat("[1/7] 读取 peaks 和准备注释...\n")

peaks_gr <- rtracklayer::import(opt$`peak-file`)
GenomeInfoDb::seqlevelsStyle(peaks_gr) <- "UCSC"
GenomeInfoDb::genome(peaks_gr) <- "hg38"
peaks_gr <- GenomeInfoDb::keepStandardChromosomes(peaks_gr, pruning.mode = "coarse")
cat("  Peaks:", length(peaks_gr), "\n")
cat("  Peak 文件读取完成\n")

annotations <- Signac::GetGRangesFromEnsDb(ensdb = EnsDb.Hsapiens.v86)
suppressWarnings(GenomeInfoDb::seqlevelsStyle(annotations) <- "UCSC")
GenomeInfoDb::genome(annotations) <- "hg38"
annotations <- GenomeInfoDb::keepStandardChromosomes(annotations, pruning.mode = "coarse")

data("blacklist_hg38_unified", package = "Signac")
suppressWarnings(GenomeInfoDb::seqlevelsStyle(blacklist_hg38_unified) <- "UCSC")
GenomeInfoDb::genome(blacklist_hg38_unified) <- "hg38"
blacklist_hg38_unified <- GenomeInfoDb::keepStandardChromosomes(blacklist_hg38_unified, pruning.mode = "coarse")

common_seq <- intersect(GenomeInfoDb::seqlevels(peaks_gr), GenomeInfoDb::seqlevels(annotations))
peaks_gr <- GenomeInfoDb::keepSeqlevels(peaks_gr, common_seq, pruning.mode = "coarse")
annotations <- GenomeInfoDb::keepSeqlevels(annotations, common_seq, pruning.mode = "coarse")
blacklist_hg38_unified <- GenomeInfoDb::keepSeqlevels(blacklist_hg38_unified, common_seq, pruning.mode = "coarse")
GenomeInfoDb::seqinfo(peaks_gr) <- GenomeInfoDb::seqinfo(annotations)[common_seq]

cat("  共同染色体数:", length(common_seq), "\n")
cat("  注释准备完成\n\n")

[1/7] 读取 peaks 和准备注释...
  Peaks: 338036 
  Peak 文件读取完成


Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warn

  共同染色体数: 24 
  注释准备完成



In [16]:
raw_dir <- file.path("data/raw", opt$gse)

barcode_files <- list.files(
  raw_dir,
  pattern = paste0("^", opt$gsm, "_.*_filtered_barcodes\\.tsv\\.gz$"),
  full.names = TRUE
)

if (length(barcode_files) == 0) {
  stop("未找到 filtered barcodes 文件: ", raw_dir)
}

barcode_file <- barcode_files[1]
barcodes <- readLines(gzfile(barcode_file))
barcodes <- barcodes[nzchar(barcodes)]

cat("  使用过滤后的 barcodes:", length(barcodes), "\n")

# ============================================================================
# 2. 创建 Fragment 对象和生成矩阵
# ============================================================================
cat("[2/7] 创建 Fragment 对象...\n")

fragment_file <- opt$`fragment-file`

# 检查 tabix 索引
tbi_file <- paste0(fragment_file, ".tbi")
if (!file.exists(tbi_file)) {
  cat("  创建 tabix 索引...\n")
  system(paste("tabix -p bed", shQuote(fragment_file)))
}

# 创建 Fragment 对象（只使用过滤后的 barcodes）
frags <- CreateFragmentObject(
  path = fragment_file,
  cells = barcodes
)
cat("  Fragment 对象创建完成\n\n")

cat("[3/7] 生成 peak×cell 计数矩阵...\n")
cat("  这可能需要几分钟...\n")

counts <- FeatureMatrix(
  fragments = frags,
  features = peaks_gr,
  cells = barcodes
)

cat("  矩阵维度:", nrow(counts), "peaks ×", ncol(counts), "cells\n\n")

  使用过滤后的 barcodes: 6685 
[2/7] 创建 Fragment 对象...
  创建 tabix 索引...


Computing hash



  Fragment 对象创建完成

[3/7] 生成 peak×cell 计数矩阵...
  这可能需要几分钟...


Extracting reads overlapping genomic regions



  矩阵维度: 338036 peaks × 6685 cells



In [17]:

# ============================================================================
# 4. 创建 Seurat 对象（按照 Signac 文档）
# ============================================================================
cat("[4/7] 创建 Seurat 对象...\n")

# 创建 ChromatinAssay
atac_assay <- CreateChromatinAssay(
  counts = counts,
  fragments = frags,
  genome = "hg38",
  sep = c(":", "-")
)

# 使用slot()直接设置annotation（绕过seqinfo错误）
slot(atac_assay, "annotation") <- annotations
cat("  注释已添加\n")

# 创建 Seurat 对象
atac_obj <- CreateSeuratObject(
  counts = atac_assay,
  assay = "ATAC",
  project = opt$gsm
)

# 添加元数据
atac_obj$sample <- opt$gsm
atac_obj$dataset <- opt$gse

cat("  Seurat 对象创建完成\n")
cat("  细胞数:", ncol(atac_obj), "\n")
cat("  特征数:", nrow(atac_obj), "\n\n")

# 保存原始 Seurat 对象（用于后续快速测试）
rds_raw <- file.path(opt$`output-dir`, paste0(opt$gsm, "_seurat_raw.rds"))
saveRDS(atac_obj, rds_raw)
cat("  原始对象已保存:", rds_raw, "\n\n")
# 结束 skip-matrix 条件块


[4/7] 创建 Seurat 对象...
  注释已添加
  Seurat 对象创建完成
  细胞数: 6685 
  特征数: 338036 

  原始对象已保存: output/GSE283744/GSM8671454_seurat_raw.rds 



In [23]:

# ============================================================================
# 5. 计算 QC 指标
# ============================================================================
cat("[5/7] 计算 QC 指标...\n")

# Nucleosome signal
cat("  计算 Nucleosome signal...\n")
atac_obj <- NucleosomeSignal(atac_obj)

# TSS enrichment
cat("  计算 TSS enrichment...\n")
atac_obj <- TSSEnrichment(atac_obj, fast = FALSE)

# 计算 FRiP 和 unique fragments ratio
cat("  计算 FRiP 和 unique ratio...\n")
total_fragments <- CountFragments(fragment_file, cells = colnames(atac_obj))
rownames(total_fragments) <- total_fragments$CB
atac_obj$fragments <- total_fragments[colnames(atac_obj), "frequency_count"]
atac_obj$total_fragments <- atac_obj$fragments
if ("reads_count" %in% colnames(total_fragments)) {
  atac_obj$reads_count <- total_fragments[colnames(atac_obj), "reads_count"]
  atac_obj$unique_ratio <- ifelse(atac_obj$reads_count > 0, atac_obj$fragments / atac_obj$reads_count, NA_real_)
}
atac_obj <- FRiP(object = atac_obj, assay = "ATAC", total.fragments = "fragments")

# 计算 blacklist fraction
cat("  计算 blacklist fraction...\n")
atac_obj$blacklist_fraction <- FractionCountsInRegion(
  object = atac_obj,
  assay = "ATAC",
  regions = blacklist_hg38_unified
)

cat("  QC 指标计算完成\n\n")


[5/7] 计算 QC 指标...
  计算 Nucleosome signal...
  计算 TSS enrichment...


Extracting TSS positions

Finding + strand cut sites

Finding - strand cut sites

Computing mean insertion frequency in flanking regions

Normalizing TSS score



  计算 FRiP 和 unique ratio...


Calculating fraction of reads in peaks per cell



  计算 blacklist fraction...
  QC 指标计算完成



In [24]:

# ============================================================================
# 6. Doublet 检测
# ============================================================================
cat("[6/7] Doublet 检测...\n")

tryCatch({
  counts_matrix <- GetAssayData(atac_obj[["ATAC"]], slot = "counts")
  sce <- SingleCellExperiment(list(counts = counts_matrix))
  sce <- scDblFinder(sce, clusters = TRUE, aggregateFeatures = TRUE,
                     nfeatures = 25, processing = "normFeatures")
  doublet_info <- as.data.frame(colData(sce)[, c("scDblFinder.class", "scDblFinder.score")])
  atac_obj$scDblFinder.class <- doublet_info$scDblFinder.class
  atac_obj$scDblFinder.score <- doublet_info$scDblFinder.score

  cat("  Singlets:", sum(doublet_info$scDblFinder.class == "singlet"), "\n")
  cat("  Doublets:", sum(doublet_info$scDblFinder.class == "doublet"), "\n\n")

}, error = function(e) {
  cat("  警告: Doublet 检测失败:", conditionMessage(e), "\n\n")
  atac_obj$scDblFinder.class <<- "unknown"
  atac_obj$scDblFinder.score <<- NA
})


[6/7] Doublet 检测...


Aggregating features...

Warning message:
"Quick-TRANSfer stage steps exceeded maximum (= 16710850)"
Clustering cells...

Warning message in (function (A, nv = 5, nu = nv, maxit = 1000, work = nv + 7, reorth = TRUE, :
"You're computing too large a percentage of total singular values, use a standard svd instead."
7 clusters

Creating ~5348 artificial doublets...

Dimensional reduction

Evaluating kNN...

Training model...

iter=0, 618 cells excluded from training.

iter=1, 621 cells excluded from training.

iter=2, 629 cells excluded from training.

Threshold found:0.767

653 (9.8%) doublets called



  Singlets: 6032 
  Doublets: 653 



In [25]:

# ============================================================================
# 7. MAD 方法过滤
# ============================================================================
cat("[7/7] 使用 MAD 方法过滤细胞...\n")

meta_data <- atac_obj@meta.data
meta_data <- subset(meta_data, nCount_ATAC > 0)
cat("  过滤 nCount_ATAC = 0:", nrow(meta_data), "cells remaining\n")


[7/7] 使用 MAD 方法过滤细胞...
  过滤 nCount_ATAC = 0: 6685 cells remaining


In [26]:

# MAD 过滤
outlier_count <- is_outlier(meta_data, "nCount_ATAC", opt$nmads)
outlier_tss <- is_outlier(meta_data, "TSS.enrichment", opt$nmads)

if ("FRiP" %in% colnames(meta_data)) {
  outlier_frip <- is_outlier(meta_data, "FRiP", opt$nmads)
  mad_outlier <- outlier_count | outlier_tss | outlier_frip
} else {
  mad_outlier <- outlier_count | outlier_tss
}

if ("scDblFinder.class" %in% colnames(meta_data)) {
  is_doublet <- meta_data$scDblFinder.class == "doublet"
  final_outlier <- mad_outlier | is_doublet
} else {
  final_outlier <- mad_outlier
}

meta_data$outlier <- final_outlier
meta_data$pass_qc <- !final_outlier

cat("  MAD 过滤前:", nrow(meta_data), "cells\n")
cat("  MAD 过滤后:", sum(!final_outlier), "cells\n")
cat("  过滤率:", sprintf("%.2f%%", sum(final_outlier) / nrow(meta_data) * 100), "\n\n")

atac_obj@meta.data <- meta_data


  MAD 过滤前: 6685 cells
  MAD 过滤后: 5756 cells
  过滤率: 13.90% 



In [27]:
# ============================================================================
# 7. QC 可视化
# ============================================================================
cat("[7/7] 生成 QC 可视化...\n")

plot_dir <- file.path(opt$`output-dir`, "qc_plots")
dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)

save_plot_png <- function(filename, width, height, expr) {
  plot_path <- file.path(plot_dir, filename)
  plot_expr <- substitute(expr)
  ok <- TRUE

  tryCatch({
    png(plot_path, width = width, height = height, res = 150)
    on.exit({
      if (dev.cur() > 1) {
        dev.off()
      }
    }, add = TRUE)
    print(eval(plot_expr, envir = parent.frame()))
  }, error = function(e) {
    ok <<- FALSE
    if (dev.cur() > 1) {
      try(dev.off(), silent = TRUE)
    }
    if (file.exists(plot_path)) {
      file.remove(plot_path)
    }
    cat("  警告:", filename, "生成失败 -", conditionMessage(e), "\n")
  })

  ok
}

# 1. QC 指标 violin plot
qc_violin_features <- c("nCount_ATAC", "nFeature_ATAC", "TSS.enrichment",
                       "FRiP", "unique_ratio", "blacklist_fraction", "nucleosome_signal")
qc_violin_features <- qc_violin_features[qc_violin_features %in% colnames(atac_obj@meta.data)]
save_plot_png(
  "01_violin_qc_metrics.png",
  1800,
  1200,
  VlnPlot(
    atac_obj,
    features = qc_violin_features,
    pt.size = 0,
    ncol = 3
  )
)

# 2. pass_qc 前后对比
qc_compare_features <- c("nCount_ATAC", "TSS.enrichment", "FRiP", "unique_ratio", "blacklist_fraction", "nucleosome_signal")
qc_compare_features <- qc_compare_features[qc_compare_features %in% colnames(atac_obj@meta.data)]
save_plot_png(
  "02_violin_pass_qc.png",
  1600,
  1000,
  VlnPlot(
    atac_obj,
    features = qc_compare_features,
    group.by = "pass_qc",
    pt.size = 0,
    ncol = 2
  )
)

# 3. 散点图
save_plot_png(
  "03_scatter_count_tss.png",
  1000,
  800,
  FeatureScatter(atac_obj, feature1 = "nCount_ATAC", feature2 = "TSS.enrichment")
)
save_plot_png(
  "04_scatter_count_frip.png",
  1000,
  800,
  FeatureScatter(atac_obj, feature1 = "nCount_ATAC", feature2 = "FRiP")
)
save_plot_png(
  "05_scatter_tss_nucleosome.png",
  1000,
  800,
  FeatureScatter(atac_obj, feature1 = "TSS.enrichment", feature2 = "nucleosome_signal")
)

# 4. TSS enrichment profile
if ("pass_qc" %in% colnames(atac_obj@meta.data)) {
  save_plot_png(
    "06_tss_plot.png",
    1000,
    800,
    TSSPlot(atac_obj, group.by = "pass_qc")
  )
}

# 5. Fragment histogram
save_plot_png(
  "07_fragment_histogram.png",
  1000,
  800,
  FragmentHistogram(atac_obj, group.by = "pass_qc")
)

# 6. doublet 结果
if ("scDblFinder.class" %in% colnames(atac_obj@meta.data)) {
  save_plot_png(
    "08_doublet_violin.png",
    1200,
    800,
    VlnPlot(
      atac_obj,
      features = c("nCount_ATAC", "nFeature_ATAC"),
      group.by = "scDblFinder.class",
      pt.size = 0,
      ncol = 2
    )
  )
}

cat("  QC 图已保存到:", plot_dir, "\n\n")

[7/7] 生成 QC 可视化...


[1] TRUE

[1] TRUE

[1] TRUE

[1] TRUE

...

[1] TRUE

Warning message:
"Removed 218 rows containing non-finite outside the scale range (`stat_bin()`)."
Warning message:
"Removed 4 rows containing missing values or values outside the scale range
(`geom_bar()`)."


[1] TRUE

[1] TRUE

  QC 图已保存到: output/GSE283744/qc_plots 

